Notebook Initialization

In [ ]:
import re
from PIL import Image, ImageFilter, ImageDraw
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import pandas as pd
import os
import scipy.stats
import collections
import matplotlib.colors as clr
import matplotlib.pylab as pl
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import norm 
from scipy.stats import gamma
import statistics
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.colors import ListedColormap
from scipy.stats import gaussian_kde
import math
from scipy.stats import pearsonr
import skimage.measure
import skimage.io
from scipy.stats import linregress
from scipy.stats import ttest_1samp, mannwhitneyu
from scipy.stats import rice, rayleigh, chi2
from scipy.special import i0
from scipy.stats import rayleigh, rice, expon, pareto, laplace
from scipy.optimize import minimize, differential_evolution
from scipy.optimize import differential_evolution, minimize
import warnings
import scipy.stats as stats
import glob
from skimage.measure import regionprops
from scipy.stats import pearsonr, circstd
from scipy.stats import binomtest
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
import pingouin as pg
from collections import defaultdict
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from skimage import measure
from matplotlib.patches import Rectangle
from pathlib import Path
from scipy.ndimage import gaussian_filter, binary_erosion


Functions and Data Initializations

In [ ]:
def blend_with(color, target=(0, 0, 0), amount=0.3):
    c = np.array(mcolors.to_rgb(color))
    target = np.array(target)
    out = (1 - amount) * c + amount * target
    return tuple(np.clip(out, 0, 1))

def make_dark_to_light_cmap(base_color, dark_amount=0.45, light_amount=0.55, name='custom_blue'):
    dark = blend_with(base_color, target=(0, 0, 0), amount=dark_amount)
    base = mcolors.to_rgb(base_color)
    light = blend_with(base_color, target=(1, 1, 1), amount=light_amount)

    cmap = LinearSegmentedColormap.from_list(
        name,
        [dark, base, light],
        N=256
    )
    return cmap

def plot_grid_timed(ids_active, df_active, ids_non, df_non,
                    fps=10, max_frame=10000,
                    cmap_active='plasma', cmap_non='viridis',
                    scale_bar_nm=100, padding=0.05):

    all_dfs = [(ids_active, df_active), (ids_non, df_non)]
    max_excursion = 0.0
    for ids, df in all_dfs:
        for pid in ids:
            traj = df[df['particle_id'] == pid]
            if traj.empty:
                continue
            x = (traj['x'].values - traj['x'].values[0]) * 2
            y = (traj['y'].values - traj['y'].values[0]) * 2
            max_excursion = max(max_excursion, np.abs(x).max(), np.abs(y).max())
    half = max_excursion * (1 + padding)

    n_active = len(ids_active)
    n_non    = len(ids_non)
    ncols    = max(n_active, n_non)

    fig, axes = plt.subplots(2, ncols,
                             figsize=(4 * ncols, 8),
                             squeeze=False)
    fig.suptitle(
        f'Trajectories centred on start  |  axis span ±{half:.0f} nm',
        fontsize=12, fontweight='bold', y=1.01
    )

    row_configs = [
        (0, ids_active, df_active, cmap_active, 'H₂O₂'),
        (1, ids_non,    df_non,    cmap_non,    'PBS'),
    ]

    for row, ids, df, cmap, condition_label in row_configs:
        for col, pid in enumerate(ids):
            ax = axes[row][col]
            traj = df[df['particle_id'] == pid].sort_values('frame').reset_index(drop=True)

            if traj.empty:
                ax.set_visible(False)
                continue

            x = (traj['x'].values - traj['x'].values[0]) * 2
            y = (traj['y'].values - traj['y'].values[0]) * 2
            t = traj['frame'].values / fps

            
            time_norm = mcolors.Normalize(vmin=t.min(), vmax=t.max())
            cmap_obj  = plt.get_cmap(cmap)

            
            for i in range(len(x) - 1):
                t_mid = (t[i] + t[i+1]) / 2
                color = cmap_obj(time_norm(t_mid))
                ax.plot(x[i:i+2], y[i:i+2],
                        color=color, lw=1.5, alpha=0.85, zorder=2,
                        solid_capstyle='round')

            
            sc = ax.scatter(x, y, c=t, cmap=cmap, norm=time_norm,
                            s=0, zorder=1)

            
            ax.set_xlim(-half,  half)
            ax.set_ylim(-half,  half)
            ax.invert_yaxis()
            ax.set_aspect('equal')
            ax.set_xticks([])
            ax.set_yticks([])

            
            xlim = ax.get_xlim()
            ylim = ax.get_ylim()
            x0 = xlim[0] + 0.06 * (xlim[1] - xlim[0])
            y0 = ylim[1] - 0.10 * (ylim[1] - ylim[0])
            ax.plot([x0, x0 + scale_bar_nm], [y0, y0],
                    color='black', lw=3, solid_capstyle='butt', zorder=5)
            ax.text(x0 + scale_bar_nm / 2, y0 - 0.03 * (ylim[1] - ylim[0]),
                    f'{scale_bar_nm} nm', ha='center', va='top', fontsize=8)

            
            cax = inset_axes(ax, width="4%", height="45%",
                             loc="upper right", borderpad=1)
            cb  = plt.colorbar(sc, cax=cax)
            cb.set_label("Time (s)", fontsize=8)
            cb.ax.tick_params(labelsize=7)

        
        for col in range(len(ids), ncols):
            axes[row][col].set_visible(False)

    plt.tight_layout()

    plt.show()

def get_msd(x):
    msd = []
    for i in range(1, len(x)):
        msd.append(np.average((x[i:] - x[:-i])**2))
    return np.array(msd)

def correct_angles(theta_degrees):
    theta_degrees = np.array(theta_degrees, dtype=float)
    n = len(theta_degrees)
    
    theta_corrected = np.zeros_like(theta_degrees)
    theta_corrected[0] = theta_degrees[0] 
    
    for t in range(n - 1):
        d = theta_degrees[t + 1] - theta_degrees[t]
        
        if -180 < d < -90:
            d_corrected = d + 180
        elif -90 <= d <= 90:
            d_corrected = d
        elif 90 < d < 180:
            d_corrected = d - 180
        else:
            d_corrected = d
        
        theta_corrected[t + 1] = theta_corrected[t] + d_corrected
    
    return theta_corrected

def get_msad(theta):
    theta = np.asarray(theta, dtype=float)
    n = len(theta)

    msad = np.empty(n - 1)
    for lag in range(1, n):
        dtheta = theta[lag:] - theta[:-lag]
        msad[lag - 1] = np.mean(dtheta**2)

    return msad

def displacement(x):
    disps = [x[i] - x[i-1] for i in range(1, len(x))]
    return disps

def r2(x,y):
    r = np.sqrt(x**2+y**2)
    return r

def plot_hist_abs(ax, dr, bins, title="", text="", color="", label="", gauss=False, guideline=True):

    dr_abs = np.abs(dr)

    counts, bins = np.histogram(dr_abs, bins=bins, density=True)
    if np.max(counts) != 0:
        counts = counts / np.max(counts)
    bin_centers = (bins[:-1] + bins[1:]) / 2

    nonzero = counts > 0
    bin_centers_plot = bin_centers[nonzero]
    counts_plot = counts[nonzero]

    ax.plot(bin_centers, counts,
            marker='o',
            linestyle='None',
            markersize=12,
            markerfacecolor='none',
            markeredgecolor=color,
            markeredgewidth=.75,
            label=label)

    if guideline:
        ax.plot(bin_centers, counts,
                linestyle='-',
                color=color,
                linewidth=4,
                alpha=0.3,
                zorder=0)

    if gauss:
        mu, sigma = scipy.stats.norm.fit(dr_abs)
        if sigma == 0:
            sigma = 1e-12

        xs = np.linspace(0, np.max(dr_abs), 1000)

        pdf = (scipy.stats.norm.pdf(xs,  mu, sigma) +
            scipy.stats.norm.pdf(xs, -mu, sigma))

        if np.max(pdf) > 0:
            pdf = pdf / np.max(pdf)

        floor = 1e-6
        pdf = np.clip(pdf, floor, None)

        ax.plot(xs, pdf,
                label=f'Folded normal (ref)',
                color='gray', linewidth=2)
        ax.fill_between(xs, pdf, floor, color='gray', alpha=0.3, zorder=0)

    return ax, bin_centers, counts

def fit_and_plot_tail(ax, bin_centers, counts, color, threshold_nm=55):
    mask = (bin_centers >= threshold_nm) & (counts > 0)
    x_tail = bin_centers[mask]
    y_tail = counts[mask]

    if len(x_tail) >= 3:
        log_x = np.log10(x_tail)
        log_y = np.log10(y_tail)
        slope, intercept, r, p, se = linregress(log_x, log_y)
        alpha = -slope

        y_pred_log = intercept + slope * log_x
        y_pred = 10**y_pred_log
        ss_res = np.sum((y_tail - y_pred)**2)
        ss_tot = np.sum((y_tail - np.mean(y_tail))**2)
        r2_linear = 1 - ss_res / ss_tot

        x_fit = np.linspace(x_tail[0], x_tail[-1], 200)
        y_fit = 10**(intercept + slope * np.log10(x_fit))
        ax.plot(x_fit, y_fit,
                linestyle='--', color=color, linewidth=2,
                label=fr'$\alpha$ = {alpha:.2f} ($R^2$={r2_linear:.2f})')
        return alpha, r2_linear
    return None, None

def compute_moment_scaling_spectrum(df, particle_col='particle_id', 
                                     q_values=[1, 2, 3, 4, 6],
                                     max_lag=30, fps=10, nm_scale=2):

    dt = 1 / fps
    lags = np.arange(1, max_lag + 1)
    time_lags = lags * dt

    moments = np.full((len(q_values), len(lags)), np.nan)

    for lag_idx, lag in enumerate(lags):
        all_disp = []

        for obj in df[particle_col].unique():
            df_obj = df[df[particle_col] == obj].dropna()

            if len(df_obj) < lag + 1:
                continue

            x = np.array(df_obj['x']) * nm_scale
            y = np.array(df_obj['y']) * nm_scale
            r = r2(x, y)

            disp = np.abs(r[lag:] - r[:-lag])
            all_disp.extend(disp)

        if len(all_disp) == 0:
            continue

        all_disp = np.array(all_disp)

        for q_idx, q in enumerate(q_values):
            moments[q_idx, lag_idx] = np.mean(all_disp ** q)


    gamma = np.full(len(q_values), np.nan)
    nu    = np.full(len(q_values), np.nan)
    r2_fits = np.full(len(q_values), np.nan)

    log_tau = np.log(time_lags)

    for q_idx, q in enumerate(q_values):
        m = moments[q_idx]
        valid = ~np.isnan(m) & (m > 0)

        if np.sum(valid) < 3:
            continue

        slope, intercept, r_val, p_val, _ = linregress(log_tau[valid], np.log(m[valid]))
        gamma[q_idx]  = slope
        nu[q_idx]     = slope / q
        r2_fits[q_idx] = r_val**2

    return {
        'q_values':  np.array(q_values),
        'gamma':     gamma,
        'nu':        nu,
        'r2_fits':   r2_fits,
        'moments':   moments,
        'time_lags': time_lags
    }

def compute_active_velocity_from_df(df, particle_id, threshold, dt, nm_per_pixel):
    g = (
        df[df['particle_id'] == particle_id]
        .sort_values('frame')
        .reset_index(drop=True)
        .copy()
    )

    if len(g) < 2:
        print(f"Skipping particle {particle_id}: too short.")
        return None

    x = g['x'].to_numpy() * nm_per_pixel
    y = g['y'].to_numpy() * nm_per_pixel

    dx = np.diff(x)
    dy = np.diff(y)
    step_sizes = np.sqrt(dx**2 + dy**2)
    valid_large = step_sizes > threshold
    velocities = step_sizes / dt

    if len(step_sizes) == 0:
        print(f"Skipping particle {particle_id}: no step sizes computed.")
        return None

    return {
        'large_step_velocities': velocities[valid_large],
        'mean_velocity_large': np.mean(velocities[valid_large]) if valid_large.sum() > 0 else np.nan
        }

def compute_speeds(df, nm_per_pixel, dt):
    df = df.copy().sort_values(['particle_id', 'frame']).reset_index(drop=True)
    speeds = np.full(len(df), np.nan)
 
    for pid, group in df.groupby('particle_id'):
        idx = group.index.values
        x = group['x'].values * nm_per_pixel
        y = group['y'].values * nm_per_pixel
 
        if len(idx) < 2:
            continue
 
        dx = np.gradient(x)
        dy = np.gradient(y)
        spd = np.sqrt(dx**2 + dy**2) / dt
        speeds[idx] = spd
 
    df['speed_nm_s'] = speeds
    return df
 
def extract_active_runs(df, threshold,
                        min_run):
    active_runs = []
    run_metadata = []
 
    for pid, group in df.groupby('particle_id'):
        group = group.sort_values('frame').reset_index(drop=True)
        speeds = group['speed_nm_s'].values
        frames = group['frame'].values
 
        is_active = speeds > threshold
 
        
        in_run = False
        run_start = 0
 
        for i in range(len(is_active)):
            if is_active[i] and not in_run:
                run_start = i
                in_run = True
            elif not is_active[i] and in_run:
                run_end = i
                run_speeds = speeds[run_start:run_end]
                if len(run_speeds) >= min_run:
                    active_runs.append(run_speeds)
                    run_metadata.append({
                        'particle_id': pid,
                        'start_frame': frames[run_start],
                        'duration_frames': run_end - run_start,
                        'mean_speed': np.mean(run_speeds),
                        'max_speed': np.max(run_speeds)
                    })
                in_run = False
 

        if in_run:
            run_speeds = speeds[run_start:]
            if len(run_speeds) >= min_run:
                active_runs.append(run_speeds)
                run_metadata.append({
                    'particle_id': pid,
                    'start_frame': frames[run_start],
                    'duration_frames': len(run_speeds),
                    'mean_speed': np.mean(run_speeds),
                    'max_speed': np.max(run_speeds)
                })
 
    return active_runs, pd.DataFrame(run_metadata)
 
def speed_autocorrelation(active_runs, max_lag):
    numerator   = defaultdict(list)
    denominator = []
 
    for run in active_runs:
        if len(run) < 2:
            continue
 
        dv = run - np.mean(run)
        var = np.mean(dv**2)
 
        if var == 0:
            continue
 
        denominator.append(var)
 
        for lag in range(0, min(max_lag + 1, len(run))):
            if lag == 0:
                c = np.mean(dv**2)
            else:
                c = np.mean(dv[:-lag] * dv[lag:])
            numerator[lag].append(c)
 
    if not denominator:
        return None, None
 
    mean_var = np.mean(denominator)
 
    lags = sorted(numerator.keys())
    acf  = []
    acf_se = []
 
    for lag in lags:
        vals = np.array(numerator[lag])
        mean_c = np.mean(vals)
        se_c   = np.std(vals, ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0
        acf.append(mean_c / mean_var)
        acf_se.append(se_c / mean_var)
 
    return np.array(lags), np.array(acf), np.array(acf_se)
 


def angular_difference(a1, a2):
    return (np.asarray(a1) - np.asarray(a2) + 180) % 360 - 180


def compute_theta_ld(x, y, window=6):
    theta = np.full(len(x), np.nan)
    half = window // 2
    for i in range(half, len(x) - half):
        dx = x[i + half] - x[i - half]
        dy = y[i + half] - y[i - half]
        if dx != 0 or dy != 0:
            theta[i] = np.degrees(np.arctan2(dy, dx))
    return theta


def compute_step_sizes(x, y):
    return np.concatenate([[np.nan], np.hypot(np.diff(x), np.diff(y))])


def _fit_circle_kasa(pts):
    x, y = pts[:, 0], pts[:, 1]
    A = np.c_[2 * x, 2 * y, np.ones(len(x))]
    b = x ** 2 + y ** 2
    a, bb, c = np.linalg.lstsq(A, b, rcond=None)[0]
    R = np.sqrt(max(c + a * a + bb * bb, 1e-9))
    resid = np.hypot(x - a, y - bb) - R
    return np.array([a, bb]), R, float(np.sqrt(np.mean(resid ** 2)))


def _fit_circle_ransac(pts, iters, thresh,
                       seed):
    n = len(pts)
    if n < 3:
        return None
    if n == 3:
        return _fit_circle_kasa(pts)
    rng = np.random.default_rng(seed)
    best_inl, best_mask = -1, None
    for _ in range(iters):
        idx = rng.choice(n, 3, replace=False)
        try:
            O, R, _ = _fit_circle_kasa(pts[idx])
        except Exception:
            continue
        if not np.isfinite(R) or R <= 0:
            continue
        d = np.abs(np.hypot(pts[:, 0] - O[0], pts[:, 1] - O[1]) - R)
        mm = d < thresh
        if mm.sum() > best_inl:
            best_inl, best_mask = mm.sum(), mm
    if best_mask is None or best_mask.sum() < 3:
        return _fit_circle_kasa(pts)
    return _fit_circle_kasa(pts[best_mask])          


def _outer_arc(mask):
    m = np.asarray(mask) > 0
    edge = m & ~binary_erosion(m)
    ys, xs = np.where(edge)
    b = np.c_[xs, ys].astype(float)
    if len(b) < 5:
        return b
    try:
        from scipy.spatial import ConvexHull
        hb = b[ConvexHull(b).vertices]
        Cc = b.mean(0)
        d = np.hypot(hb[:, 0] - Cc[0], hb[:, 1] - Cc[1])
        outer = hb[d > 0.6 * d.mean()]
        return outer if len(outer) >= 3 else hb
    except Exception:
        return b


def get_pt_angle_from_mask(mask, iters, thresh, seed, min_area):
    m = np.asarray(mask) > 0
    props = regionprops(m.astype(int))
    if not props:
        return np.nan, np.nan, np.nan
    cy, cx = props[0].centroid
    if m.sum() < min_area:
        return np.nan, np.nan, np.nan

    fit = _fit_circle_ransac(_outer_arc(m),iters, thresh, seed)
    if fit is None:
        return np.nan, np.nan, np.nan
    O, R, rms = fit
    if not np.isfinite(R) or R <= 0:
        return np.nan, np.nan, np.nan

    ox, oy = cx - O[0], cy - O[1]
    return (float(np.degrees(np.arctan2(oy, ox))),
            float(np.hypot(ox, oy)), float(R))


def index_dir(d, patterns, obj=None):
    out = {}
    for pat in patterns:
        for f in sorted(glob.glob(os.path.join(d, pat))):
            base = os.path.splitext(os.path.basename(f))[0]
            mm = re.match(r'^(\d+)', base)
            if not mm:
                continue
            if obj is not None:
                mo = re.search(r'obj(\d+)', base)
                if mo is None or int(mo.group(1)) != obj:
                    continue
            out[int(mm.group(1))] = f
    return out


def analyze_folder(folder_name, iters, thresh, seed, min_area, particle_id=None,):
    pf = os.path.join(ROOT, folder_name)
    if particle_id is None:
        particle_id = 0

    msk_map = index_dir(os.path.join(pf, MSK_DIRNAME), ["*.png"])

    df = pd.read_csv(os.path.join(pf, "trajectories.csv"))
    df = df[(df['frame'] % SUBSAMPLE == 0) & (df['particle_id'] == particle_id)]
    df = df.drop_duplicates('frame').sort_values('frame').reset_index(drop=True)
    if df.empty:
        return None

    xs, ys, fs, angs, offs, rads = [], [], [], [], [], []
    n_no_mask = n_small = 0
    for _, row in df.iterrows():
        fr = int(row['frame'])
        if fr not in msk_map:
            n_no_mask += 1
            continue                    
        mk = np.array(Image.open(msk_map[fr]).convert('L')) > 0
        if mk.sum() == 0:
            n_no_mask += 1
            continue
        a, o, R = get_pt_angle_from_mask(mk,iters, thresh, seed, min_area)
        if np.isnan(a):
            n_small += 1
        angs.append(a); offs.append(o); rads.append(R)
        xs.append(float(row['x'])); ys.append(float(row['y'])); fs.append(fr)

    if not fs:
        return None

    x = np.array(xs); y = np.array(ys); frames = np.array(fs, int)
    pt_angles = np.array(angs, float)
    pt_offset = np.array(offs, float)
    pt_R      = np.array(rads, float)

    theta_ld = compute_theta_ld(x, y, window=WINDOW_LD)
    steps    = compute_step_sizes(x, y)
    valid_large = (steps > THRESHOLD_PX) & ~np.isnan(pt_angles) \
                  & ~np.isnan(theta_ld)

    diffs = angular_difference(pt_angles[valid_large], theta_ld[valid_large])
    cos_vals = np.cos(np.radians(diffs)) if len(diffs) else np.array([])
    cos_mean = float(cos_vals.mean()) if len(cos_vals) else np.nan
    cos_se = (float(cos_vals.std(ddof=1) / np.sqrt(len(cos_vals)))
              if len(cos_vals) > 1 else np.nan)
    verdict = ("No data" if np.isnan(cos_mean) else
               "Pt FORWARD" if cos_mean > 0.1 else
               "Pt TRAILING" if cos_mean < -0.1 else "No preference")

    return dict(label=folder_name, frames=frames, x=x, y=y,
                pt_angles=pt_angles, pt_offset=pt_offset, pt_R=pt_R,
                theta_ld=theta_ld, steps=steps, valid_large=valid_large,
                dtheta=diffs, cos_vals=cos_vals, cos_large=cos_mean,
                cos_se=cos_se, verdict=verdict, n_large=int(len(cos_vals)),
                n_traj=len(df), n_kept=len(fs),
                n_no_mask=n_no_mask, n_small=n_small)

def vtest(angles_rad, mu0):
    m = len(angles_rad)
    V = float(np.sum(np.cos(angles_rad - mu0)))
    u = V * np.sqrt(2.0 / m)
    return V, u, float(stats.norm.sf(u))

In [ ]:
blue_color = '#3C6578'
purple_color = '#F08686'
blue_cmap = make_dark_to_light_cmap(blue_color)
purple_cmap = make_dark_to_light_cmap(purple_color)

figure_output = r'\PtPS Figures'

In [ ]:
df_active = pd.read_csv(r'H202\H2O2.csv')
df_non = pd.read_csv(r'PBS\PBS.csv')


df_active = df_active.sort_values(by=['particle_id', 'frame']).reset_index(drop=True)
df_non = df_non.sort_values(by=['particle_id', 'frame']).reset_index(drop=True)

selected_active_ids = [0,1,2,3,4,5]

df_active = df_active[df_active['particle_id'].isin(selected_active_ids)].reset_index(drop=True)

non_ids=[0,1,2,3,4,5]

FPS = 10
MAX_FRAME = 1000

Transport Statistics

In [ ]:
plot_grid_timed(
    ids_active   = selected_active_ids,
    df_active    = df_active,
    ids_non      = non_ids,
    df_non       = df_non,
    fps          = FPS,
    max_frame    = MAX_FRAME,
    cmap_active  = blue_cmap,
    cmap_non     = purple_cmap,
    scale_bar_nm = 100,
)


In [ ]:
fps = 10
all_msd = []
all_time = None

for obj in df_non['particle_id'].unique():
    df_non_obj = df_non[df_non['particle_id'] == obj]
    df_non_obj = df_non_obj.dropna()
    x = np.array(df_non_obj['x'])*2
    y = np.array(df_non_obj['y'])*2
    frame = np.array(df_non_obj['frame'])
    angle = np.array(df_non_obj['angle'])
    msd1_non = get_msd(np.sqrt(x**2 + y**2))
    all_msd.append(msd1_non)

    time_non = [1/fps]
    for i in range(1, len(df_non_obj)):
        time_non.append(time_non[i-1] + 1/fps)
    angle_time_non = time_non
    time_non = time_non[:len(x)-1]

    if all_time is None or len(time_non) > len(all_time):
        all_time = time_non

if all_msd and all_time is not None:
    min_len_non = min(len(m) for m in all_msd)
    etmsd_non = np.mean([m[:min_len_non] for m in all_msd], axis=0)
    etmsd_time_non = all_time[:min_len_non]


fps = 10
selected_ids = [0,1,2,3,4,5]
all_msd_active = []
all_time_active = None

for obj in selected_ids:
    df_obj = df_active[df_active['particle_id'] == obj].dropna()
    
    if len(df_obj) == 0:
        print(f"Warning: particle_id {obj} not found, skipping.")
        continue

    x = np.array(df_obj['x'])*2
    y = np.array(df_obj['y'])*2

    msd_active = get_msd(np.sqrt(x**2 + y**2))
    all_msd_active.append(msd_active)

    time_active = [i / fps for i in range(1, len(df_obj))]

    if all_time_active is None or len(time_active) > len(all_time):
        all_time_active = time_active

if all_msd_active and all_time_active is not None:
    min_len_active = min(len(m) for m in all_msd_active)
    etmsd_active = np.mean([m[:min_len_active] for m in all_msd_active], axis=0)
    etmsd_time_active = all_time_active[:min_len_active]

In [ ]:
slope1_non, intercept1_non = np.polyfit(np.log10(etmsd_time_non[0:30]), np.log10(etmsd_non[0:30]),1)
print(slope1_non)
print((10**intercept1_non)/4)

slope1_active, intercept1_active = np.polyfit(np.log10(etmsd_time_active[0:5]), np.log10(etmsd_active[0:5]),1)
print(slope1_active)
print(intercept1_active)
print((10**intercept1_active)/4)

slope2_active, intercept2_active = np.polyfit(np.log10(etmsd_time_active[11:25]), np.log10(etmsd_active[11:25]),1)
print(slope2_active)

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))

ax.plot(etmsd_time_active, etmsd_active, color=blue_color, lw=5, label=f'etMSD (n={len(all_msd_active)})')
ax.plot(etmsd_time_active[11:45], (10**intercept2_active) * np.array(etmsd_time_active[11:45])**slope2_active,
            color='gray', label=f'⍺={slope1_active:.3f}',zorder=7,lw=2)
ax.plot(etmsd_time_active[0:6], (10**intercept1_active) * np.array(etmsd_time_active[0:6])**slope1_active,
            color='gray', label=f'⍺={slope1_active:.3f}',zorder=7,lw=2)

ax.plot(etmsd_time_non, etmsd_non, color=purple_color, lw=5)
ax.plot(etmsd_time_non[0:25], (10**intercept1_non) * np.array(etmsd_time_non[0:25])**slope1_non,
            color='gray', label=f'⍺={slope1_non:.3f}',zorder=7,lw=2)

ax.axvline(x=1.0, color='gray', linestyle='--', lw=1)

ax.set_yscale('log')
ax.set_xscale('log')

ax.set_xlabel(r'$\tau$ (s)', fontsize=20)
ax.set_ylabel(r'$\overline{\delta r^2(\tau)}$ (nm$^2$)', fontsize=20)
plt.yticks(fontsize=20)
plt.xticks(fontsize=20)
plt.tight_layout()
plt.show()

In [ ]:
df_active_sub10 = df_active.groupby('particle_id').apply(lambda g: g.iloc[::10]).reset_index(drop=True)
df_non_sub10 = df_non.groupby('particle_id').apply(lambda g: g.iloc[::10]).reset_index(drop=True)

In [ ]:
fps = 1
selected_ids_sub10_active = [0,1,2,3,4,5]
all_msd_active_sub10 = []
all_time_active_sub10 = None

for obj in selected_ids_sub10_active:
    df_obj = df_active_sub10[df_active_sub10['particle_id'] == obj].dropna()
    
    if len(df_obj) == 0:
        print(f"Warning: particle_id {obj} not found, skipping.")
        continue

    x = np.array(df_obj['x'])*2
    y = np.array(df_obj['y'])*2

    msd_active_sub10 = get_msd(np.sqrt(x**2 + y**2))
    all_msd_active_sub10.append(msd_active_sub10)

    time_active_sub10 = [i / fps for i in range(1, len(df_obj))]

    if all_time_active_sub10 is None or len(time_active_sub10) > len(all_time_active_sub10):
        all_time_active_sub10 = time_active_sub10

if all_msd_active_sub10 and all_time_active_sub10 is not None:
    min_len_active_sub10 = min(len(m) for m in all_msd_active_sub10)
    etmsd_active_sub10 = np.mean([m[:min_len_active_sub10] for m in all_msd_active_sub10], axis=0)
    etmsd_time_active_sub10 = all_time_active_sub10[:min_len_active_sub10]

fps = 1

all_msd_non_sub10 = []
all_time_non_sub10 = None
selected_ids_sub10_non = [0,1,2,3,4,5]

for obj in selected_ids_sub10_non:
    df_obj = df_non_sub10[df_non_sub10['particle_id'] == obj].dropna()
    
    if len(df_obj) == 0:
        print(f"Warning: particle_id {obj} not found, skipping.")
        continue

    x = np.array(df_obj['x'])*2
    y = np.array(df_obj['y'])*2

    msd_non_sub10 = get_msd(np.sqrt(x**2 + y**2))
    all_msd_non_sub10.append(msd_non_sub10)

    time_non_sub10 = [i / fps for i in range(1, len(df_obj))]

    if all_time_non_sub10 is None or len(time_non_sub10) > len(all_time_non_sub10):
        all_time_non_sub10 = time_non_sub10


if all_msd_non_sub10 and all_time_non_sub10 is not None:
    min_len_non_sub10 = min_len_active_sub10
    etmsd_non_sub10 = np.mean([m[:min_len_non_sub10] for m in all_msd_non_sub10], axis=0)
    etmsd_time_non_sub10 = all_time_non_sub10[:min_len_non_sub10]


slope1_active_sub10, intercept1_active_sub10 = np.polyfit(np.log10(etmsd_time_active_sub10[0:24]), np.log10(etmsd_active_sub10[0:24]),1)

slope1_non_sub10, intercept1_non_sub10 = np.polyfit(np.log10(etmsd_time_non_sub10[0:4]), np.log10(etmsd_non_sub10[0:4]),1)

In [ ]:
fig, ax = plt.subplots(figsize=(6,4.5))

ax.scatter(etmsd_time_active_sub10[0:30], etmsd_active_sub10[0:30], color=blue_color, s=20,zorder=5, label=f'etMSD (n={len(all_msd_active_sub10)})')
ax.plot(etmsd_time_active_sub10[0:6], (10**intercept1_active_sub10) * np.array(etmsd_time_active_sub10[0:6])**slope1_active_sub10,
            color='gray', label=f'⍺={slope1_active_sub10:.3f}',zorder=7)

ax.scatter(etmsd_time_non_sub10[0:30], etmsd_non_sub10[0:30], color=purple_color, s=20,zorder=5, label=f'etMSD (n={len(all_msd_non_sub10)})')
ax.plot(etmsd_time_non_sub10[0:6], (10**intercept1_non_sub10) * np.array(etmsd_time_non_sub10[0:6])**slope1_non_sub10,
            color='gray', label=f'⍺={slope1_non_sub10:.3f}',zorder=7)



for obj in selected_ids_sub10_active:
    df_obj = df_active_sub10[df_active_sub10['particle_id'] == obj].dropna()
    
    if len(df_obj) == 0:
        print(f"Warning: particle_id {obj} not found, skipping.")
        continue

    x = np.array(df_obj['x'])*2
    y = np.array(df_obj['y'])*2

    msd_active_sub10 = get_msd(np.sqrt(x**2 + y**2))
    all_msd_active_sub10.append(msd_active_sub10)

    time_active_sub10 = [i / fps for i in range(1, len(df_obj))]

    if all_time_active_sub10 is None or len(time_active_sub10) > len(all_time_active_sub10):
        all_time_active_sub10 = time_active_sub10

    
    ax.plot(time_active_sub10[0:30], msd_active_sub10[0:30], color=blue_color, alpha=0.15, lw =2)

for obj in selected_ids_sub10_non:
    df_obj = df_non_sub10[df_non_sub10['particle_id'] == obj].dropna()
    
    if len(df_obj) == 0:
        print(f"Warning: particle_id {obj} not found, skipping.")
        continue

    x = np.array(df_obj['x'])*2
    y = np.array(df_obj['y'])*2

    msd_non_sub10 = get_msd(np.sqrt(x**2 + y**2))
    all_msd_non_sub10.append(msd_non_sub10)

    time_non_sub10 = [i / fps for i in range(1, len(df_obj))]

    if all_time_non_sub10 is None or len(time_non_sub10) > len(all_time_non_sub10):
        all_time_non_sub10 = time_non_sub10

    
    ax.plot(time_non_sub10[0:30], msd_non_sub10[0:30], color=purple_color, alpha = 0.15, lw=2)

ax.set_yscale('log')
ax.set_xscale('log')
ax.set_xlabel(r'$\tau$ (s)', fontsize=12)
ax.set_ylabel(r'$\overline{\delta r^2(\tau)}$ (nm$^2$)', fontsize=12)
ax.set_xlim(.96,26)
plt.yticks(fontsize=12)
plt.xticks(fontsize=12)
plt.tight_layout()



plt.show()

In [ ]:
df_active_sub10 = df_active_sub10.sort_values(['particle_id', 'frame']).copy()
df_active_sub10['angle_corrected'] = (
    df_active_sub10.groupby('particle_id')['angle']
    .transform(lambda s: pd.Series(correct_angles(s.to_numpy()), index=s.index))
)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))

all_msad_active_sub10 = []
all_time_active_sub10 = None

for obj in selected_ids_sub10_active:
    df_obj = (
        df_active_sub10[df_active_sub10['particle_id'] == obj]
        .dropna(subset=['angle_corrected'])
        .sort_values('frame')
    )

    if len(df_obj) < 2:
        print(f"Warning: particle_id {obj} has fewer than 2 angle points, skipping.")
        continue

    theta = np.array(df_obj['angle_corrected'], dtype=float)

    msad_active_sub10 = get_msad(theta)
    all_msad_active_sub10.append(msad_active_sub10)

    time_active_sub10 = np.arange(1, len(theta)) / fps

    if all_time_active_sub10 is None or len(time_active_sub10) > len(all_time_active_sub10):
        all_time_active_sub10 = time_active_sub10

    
    ax.plot(time_active_sub10, msad_active_sub10, color=blue_color, alpha=0.05)


if all_msad_active_sub10 and all_time_active_sub10 is not None:
    etmsad_active_sub10 = np.mean(
        [m[:min_len_active_sub10] for m in all_msad_active_sub10],
        axis=0
    )
    etmsad_time_active_sub10 = all_time_active_sub10[:min_len_active_sub10]

    ax.scatter(
        etmsad_time_active_sub10,
        etmsad_active_sub10,
        color=blue_color,
        s=20,
        zorder=5,
        label=f'etMSAD (n={len(all_msad_active_sub10)})'
    )

    ax.scatter(
        etmsad_time_active_sub10,
        etmsad_active_sub10,
        color='white',
        s=3,
        zorder=9,
        label=f'etMSAD (n={len(all_msad_active_sub10)})'
    )
    

ax.set_yscale('log')
ax.set_xscale('log')
ax.set_xlim(0,26)
ax.set_xlabel(r'$\tau$ (s)', fontsize=12)
ax.set_ylabel(r'$\overline{\delta r^2(\tau)}$ (nm$^2$)', fontsize=12)
plt.yticks(fontsize=12)
plt.xticks(fontsize=12)
plt.tight_layout()


plt.show()

In [ ]:
df_active_sub10_selected = df_active_sub10[
    df_active_sub10['particle_id'].isin(selected_ids_sub10_active)
].reset_index(drop=True)

all_disp_active_sub10 = []

for obj in selected_ids_sub10_active:
    df_obj = df_active_sub10_selected[df_active_sub10_selected['particle_id'] == obj].dropna()
    x = np.array(df_obj['x']) *2
    y = np.array(df_obj['y'])*2
    r = r2(x, y)
    disp_r = displacement(r)
    all_disp_active_sub10.append(disp_r)

all_disp_active_sub10 = np.concatenate(all_disp_active_sub10)

df_non_sub10_selected = df_non_sub10[
    df_non_sub10['particle_id'].isin(selected_ids_sub10_non)
].reset_index(drop=True)

all_disp_non_sub10 = []
for obj in selected_ids_sub10_non:
    df_obj = df_non_sub10_selected[df_non_sub10_selected['particle_id'] == obj].dropna()
    x = np.array(df_obj['x'])*2 
    y = np.array(df_obj['y'])*2
    r = r2(x, y)
    disp_r = displacement(r)
    all_disp_non_sub10.append(disp_r)

all_disp_non_sub10 = np.concatenate(all_disp_non_sub10)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))

ax, bc_pbs,  ct_pbs  = plot_hist_abs(ax, all_disp_non_sub10, bins=12, color=purple_color,
                                      label='2.5mM PBS',  guideline=True, gauss=True)
ax, bc_h2o2, ct_h2o2 = plot_hist_abs(ax, all_disp_active_sub10, bins=13, color=blue_color,
                                      label='0.5% H₂O₂', guideline=True, gauss=False)

fit_and_plot_tail(ax, bc_pbs,  ct_pbs,  color=purple_color, threshold_nm=75)
fit_and_plot_tail(ax, bc_h2o2, ct_h2o2, color=blue_color,   threshold_nm=60)

ax.set_ylim(0.003, 1.01)
ax.legend()
ax.set_xlim(0, 263)
ax.set_yscale('log')
plt.tight_layout()

In [ ]:
fps_sub10 = 1
dt_sub10  = 1 / fps_sub10

mss_active_s10  = compute_moment_scaling_spectrum(
    df_active_sub10,
    particle_col='particle_id',
    q_values=[1, 2, 3, 4, 5, 6],
    max_lag=4, 
    fps=fps_sub10,
    nm_scale=2
)

mss_passive_s10 = compute_moment_scaling_spectrum(
    df_non_sub10,
    particle_col='particle_id',
    q_values=[1, 2, 3, 4, 5, 6],
    max_lag=7,
    fps=fps_sub10,
    nm_scale=2
)


nu_std_active  = np.nanstd(mss_active_s10['nu'])
nu_std_passive = np.nanstd(mss_passive_s10['nu'])

fig, ax = plt.subplots(figsize=(6, 4.5))

q_vals = mss_active_s10['q_values']
q_fine = np.linspace(q_vals[0], q_vals[-1], 200)

ax.plot(q_fine, np.ones_like(q_fine) * 0.5, 'k--',
        linewidth=1.5, label='Brownian ν=0.5', zorder=1)
ax.plot(q_vals, mss_active_s10['nu'],  's-',
        lw=2, markersize=8, label=f'H₂O₂ sub10 (std={nu_std_active:.3f})',  zorder=3,color=blue_color)
ax.plot(q_vals, mss_passive_s10['nu'], 's-',
        lw=2, markersize=8, label=f'PBS sub10  (std={nu_std_passive:.3f})', zorder=3,color=purple_color)
ax.set_xlabel('Moment order q', fontsize=12)
ax.set_ylabel('ν(q)', fontsize=12)
ax.legend(fontsize=10)
ax.tick_params(labelsize=12)
ax.set_xticks(q_vals)

plt.tight_layout()

plt.show()

In [ ]:
THRESHOLD = 30.7 
FPS = 1
DT = 1 / FPS
NM_PER_PIXEL = 2

velocity_results = []

for particle_id in selected_active_ids:
    res = compute_active_velocity_from_df(df_active_sub10, particle_id, threshold = THRESHOLD, dt=DT, nm_per_pixel=NM_PER_PIXEL)
    if res is not None:
        velocity_results.append(res)

all_large_velocities_list = [
    r['large_step_velocities']
    for r in velocity_results
    if len(r['large_step_velocities']) > 0
]

if len(all_large_velocities_list) > 0:
    all_large_velocities = np.concatenate(all_large_velocities_list)
    prop_speed = np.mean(all_large_velocities)
    print(f"Average Propulsion Speed (nm/s): {prop_speed:.1f}")
else:
    print("No large steps found above threshold.")


particle_diameter_nm = 110
D_eff_pbs            = 235
v_propulsion         = prop_speed

body_lengths_per_s = v_propulsion / particle_diameter_nm

dt = 1
v_thermal_expected = np.sqrt(4 * D_eff_pbs / dt)

D_translational = D_eff_pbs 
Pe = v_propulsion * (particle_diameter_nm/2) / D_translational


print(f"Pe = {Pe:.0f}")

In [ ]:
SPEED_THRESHOLD_NM_S = 30.7
NM_PER_PIXEL = 2.0
DT_SECONDS = 1
MIN_RUN_LENGTH = 3
MAX_LAG = 3
MIN_RUN=3
df = df_active_sub10_selected
 
df = compute_speeds(df,nm_per_pixel=NM_PER_PIXEL,dt=DT)
active_frames_only = df[df['speed_nm_s'] > SPEED_THRESHOLD_NM_S]

active_runs, run_meta = extract_active_runs(df,threshold=SPEED_THRESHOLD_NM_S,
                        min_run=MIN_RUN_LENGTH)

lags, acf, acf_se = speed_autocorrelation(active_runs, max_lag=MAX_LAG)
 
fig, ax = plt.subplots(figsize=(6, 4.5))
 
if lags is not None:
    t_lags = lags * DT_SECONDS
    ax.plot(t_lags, acf,color=blue_color, lw=2, label='ACF')
    ax.scatter(t_lags, acf,color=blue_color, marker='d', label='ACF')
    ax.axhline(0, color='gray', lw=1, linestyle=':')

ax.set_xlabel('Lag time (s)', fontsize=10)
ax.set_ylabel('C(τ)', fontsize=10)
 
plt.tight_layout()

Polarity Analysis

In [ ]:
ROOT        = r"\H2O2"
FOLDERS     = sorted([d for d in os.listdir(ROOT)
                      if os.path.isdir(os.path.join(ROOT, d)) and d.isdigit()],
                     key=int)
MSK_DIRNAME = "masks"
MIN_AREA    = 1000
SUBSAMPLE     = 10
WINDOW_LD     = 6
D_EFF         = 235       
LAG_TIME_S    = 1
MULTIPLIER    = 1
PX_NM         = 2.0

THRESHOLD_NM  = MULTIPLIER * np.sqrt(4 * D_EFF * LAG_TIME_S)
THRESHOLD_PX  = THRESHOLD_NM / PX_NM      
RANSAC_ITERS  = 300
RANSAC_THRESH = 2.0
RANSAC_SEED   = 0

results = []
for fn in FOLDERS:
    r = analyze_folder(fn, iters=RANSAC_ITERS, thresh = RANSAC_THRESH, seed = RANSAC_SEED, min_area = MIN_AREA)
    if r is None:
        continue
    results.append(r)

cos_pooled = np.concatenate([r['cos_vals'] for r in results
                             if len(r['cos_vals'])])
off_pooled = np.concatenate([r['pt_offset'][~np.isnan(r['pt_offset'])]
                             for r in results])
R_pooled = np.concatenate([r['pt_R'][~np.isnan(r['pt_R'])] for r in results])

print(f"mean cos = {cos_pooled.mean():+.2f}, "
      f"median = {np.median(cos_pooled):+.2f}")

In [ ]:
FWD_COLOR   = '#4C69D2'
TRAIL_COLOR = '#e74c3c'
PT_COLOR    = '#C8A2E2'

diffs_all = []
for r in results:
    v = r['valid_large']
    if v.sum() == 0:
        continue
    diffs_all.extend(angular_difference(r['pt_angles'][v], r['theta_ld'][v]))
diffs_all = np.asarray(diffs_all, float)
diffs_rad = np.radians(diffs_all)

cos_vals_h2o2 = np.concatenate([r['cos_vals'] for r in results
                                if len(r['cos_vals']) > 0])
cos_vals = cos_vals_h2o2
n = len(diffs_rad)

C      = np.mean(np.cos(diffs_rad))
S      = np.mean(np.sin(diffs_rad))
R      = np.sqrt(C ** 2 + S ** 2)
mu_rad = np.arctan2(S, C)

fig = plt.figure(figsize=(5, 5))
fig.patch.set_facecolor('white')
ax = fig.add_subplot(111, projection='polar')
n_bins      = 20 
bin_edges   = np.linspace(-np.pi, np.pi, n_bins + 1)
counts, _   = np.histogram(diffs_rad, bins=bin_edges)
bin_width   = 2 * np.pi / n_bins
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
counts_norm = counts / counts.max()
bar_colors = [TRAIL_COLOR if abs(np.degrees(bc)) > 90
              else FWD_COLOR for bc in bin_centers]
ax.bar(bin_centers, counts_norm,
       width=bin_width * 0.9,
       color=bar_colors, alpha=0.75,
       edgecolor='white', linewidth=0.5)
ax.annotate('',
    xy=(mu_rad, 0.8),
    xytext=(0, 0),
    arrowprops=dict(arrowstyle='->', color='black', lw=2.5, zorder=6))
ax.set_theta_zero_location('E')
ax.set_theta_direction(1)
ax.set_xticks(np.radians([0, 90, 180, 270]))
ax.set_xticklabels(['0°', '90°', '±180°', '-90°'],
                    fontsize=9, fontweight='bold')
ax.set_yticklabels([])
theta_trail = np.linspace(np.pi/2, 3*np.pi/2, 100)
ax.fill_between(theta_trail, 0, 1.01,
                alpha=0.05, color=TRAIL_COLOR, zorder=0)
ax.set_ylim(0, 1.01)
ax.text(mu_rad, 0.88,
        f'{np.degrees(mu_rad):.1f}°',
        ha='center', fontsize=8, color='black', fontweight='bold')
plt.tight_layout()
plt.show()

ROSE_NORM      = int(counts.max())
ROSE_N_BINS    = n_bins
ROSE_BIN_EDGES = bin_edges
ROSE_COUNTS    = counts


bin_edges   = np.linspace(-1, 1, 11)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width   = bin_edges[1] - bin_edges[0]
counts_h2o2, _ = np.histogram(cos_vals_h2o2, bins=bin_edges)
global_max     = counts_h2o2.max()
density_h2o2   = counts_h2o2 / global_max

fig, ax1 = plt.subplots(figsize=(4, 4.5))
fig.patch.set_facecolor('white')
bar_colors = [FWD_COLOR if bc > 0 else TRAIL_COLOR for bc in bin_centers]
ax1.bar(bin_centers, density_h2o2, width=bin_width * 0.92,
        color=bar_colors, alpha=0.78, edgecolor='white', lw=0.5)
ax1.set_xlim(-1, 1)
ax1.set_ylim(0, 1.02)
ax1.axvspan(-1, 0, alpha=0.06, color=TRAIL_COLOR, zorder=0)
ax1.axvspan( 0, 1, alpha=0.06, color=FWD_COLOR,   zorder=0)
ax1.set_xlabel('cos(θ$_{Pt}$ − θ$_{ld}$)', fontsize=10)
ax1.set_ylabel('PDF', fontsize=10)
ax1.tick_params(labelsize=8)
plt.tight_layout()
plt.show()

PDF_NORM      = int(counts_h2o2.max())
PDF_BIN_EDGES = bin_edges
PDF_COUNTS    = counts_h2o2


print(f"mean resultant vector = {np.degrees(mu_rad):+.2f} deg")
print(f"mean cos = {cos_pooled.mean():+.2f}")


In [ ]:
cos_all = np.asarray(cos_pooled, float)
n = cos_all.size
dth_rad = np.radians(np.concatenate(
    [np.asarray(r['dtheta'], float) for r in results if len(r['dtheta'])]))

mean_c = float(cos_all.mean())
sd_c   = float(cos_all.std(ddof=1))
sem_c  = sd_c / np.sqrt(n)
t_stat, p_t = stats.ttest_1samp(cos_all, 0.0, alternative='less')

tcrit = stats.t.ppf(0.975, n - 1)
ci_lo, ci_hi = mean_c - tcrit * sem_c, mean_c + tcrit * sem_c

V, u, p_v = vtest(dth_rad, np.pi)
Rbar   = float(np.abs(np.mean(np.exp(1j * dth_rad))))
mu_dir = float(np.degrees(np.angle(np.mean(np.exp(1j * dth_rad)))))

print(f"mean cos = {mean_c:+.2f}")
print(f"95% CI on mean = [{ci_lo:+.2f}, {ci_hi:+.2f}]")
print(f"one-sided t-test (H1: mean cos < 0): p = {p_t:.3f}")

print(f"mean direction = {mu_dir:+.1f}°")
print(f"V-test toward {np.degrees(np.pi):.0f}°: p = {p_v:.3f}")

In [ ]:
ROOT        = r"\PBS"
FOLDERS     = sorted([d for d in os.listdir(ROOT)
                      if os.path.isdir(os.path.join(ROOT, d)) and d.isdigit()],
                     key=int)
MSK_DIRNAME = "masks"
MIN_AREA    = 1000
SUBSAMPLE     = 10
WINDOW_LD     = 6
D_EFF         = 235      
LAG_TIME_S    = 1
MULTIPLIER    = 1
PX_NM         = 2.0

THRESHOLD_NM  = MULTIPLIER * np.sqrt(4 * D_EFF * LAG_TIME_S)
THRESHOLD_PX  = THRESHOLD_NM / PX_NM      
RANSAC_ITERS  = 300
RANSAC_THRESH = 2.0
RANSAC_SEED   = 0

pbs_results = []
for fn in FOLDERS:
    r = analyze_folder(fn, iters=RANSAC_ITERS, thresh = RANSAC_THRESH, seed = RANSAC_SEED, min_area = MIN_AREA)
    if r is None:
        continue
    pbs_results.append(r)

pbs_cos_pooled = np.concatenate([r['cos_vals'] for r in pbs_results
                             if len(r['cos_vals'])])

print(f"mean cos = {pbs_cos_pooled.mean():+.3f}, "
      f"median = {np.median(pbs_cos_pooled):+.3f}")

In [ ]:
ROSE_NORM   = 7        
ROSE_N_BINS = 20
PDF_NORM    = 11       

RESULTS_pbs = globals().get('results_pbs', pbs_results)

diffs_all = []
for r in RESULTS_pbs:
    v = r['valid_large']
    if v.sum() == 0:
        continue
    diffs_all.extend(angular_difference(r['pt_angles'][v], r['theta_ld'][v]))
diffs_all = np.asarray(diffs_all, float)
diffs_rad = np.radians(diffs_all)

cos_vals_pbs = np.concatenate([r['cos_vals'] for r in RESULTS_pbs
                               if len(r['cos_vals']) > 0])
cos_vals = cos_vals_pbs
n = len(diffs_rad)

C      = np.mean(np.cos(diffs_rad))
S      = np.mean(np.sin(diffs_rad))
R      = np.sqrt(C ** 2 + S ** 2)
mu_rad_pbs = np.arctan2(S, C)


fig = plt.figure(figsize=(5, 5))
fig.patch.set_facecolor('white')
ax = fig.add_subplot(111, projection='polar')
n_bins      = ROSE_N_BINS
bin_edges   = np.linspace(-np.pi, np.pi, n_bins + 1)
counts, _   = np.histogram(diffs_rad, bins=bin_edges)
bin_width   = 2 * np.pi / n_bins
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

counts_norm = counts / ROSE_NORM
bar_colors = [TRAIL_COLOR if abs(np.degrees(bc)) > 90
              else FWD_COLOR for bc in bin_centers]
ax.bar(bin_centers, counts_norm,
       width=bin_width * 0.9,
       color=bar_colors, alpha=0.75,
       edgecolor='white', linewidth=0.5)

ax.annotate('',
    xy=(mu_rad_pbs, 0.8),
    xytext=(0, 0),
    arrowprops=dict(arrowstyle='->', color='black', lw=2.5, zorder=6))

ax.set_theta_zero_location('E')
ax.set_theta_direction(1)
ax.set_xticks(np.radians([0, 90, 180, 270]))
ax.set_xticklabels(['0°', '90°', '±180°', '-90°'],
                    fontsize=9, fontweight='bold')
ax.set_yticklabels([])
theta_trail = np.linspace(np.pi/2, 3*np.pi/2, 100)
ax.fill_between(theta_trail, 0, 1.01,
                alpha=0.05, color=TRAIL_COLOR, zorder=0)
ax.set_ylim(0, 1.01)
ax.text(mu_rad_pbs, 0.88,
        f'μ={np.degrees(mu_rad_pbs):.1f}°',
        ha='center', fontsize=8, color='black', fontweight='bold')
plt.tight_layout()
plt.show()


bin_edges   = np.linspace(-1, 1, 11)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width   = bin_edges[1] - bin_edges[0]
counts_pbs, _ = np.histogram(cos_vals_pbs, bins=bin_edges)
density_pbs   = counts_pbs / PDF_NORM

fig, ax1 = plt.subplots(figsize=(4, 4.5))
fig.patch.set_facecolor('white')
bar_colors = [FWD_COLOR if bc > 0 else TRAIL_COLOR for bc in bin_centers]
ax1.bar(bin_centers, density_pbs, width=bin_width * 0.92,
        color=bar_colors, alpha=0.78, edgecolor='white', lw=0.5)
ax1.set_xlim(-1, 1.02)
ax1.set_ylim(0, 1)
ax1.axvspan(-1, 0, alpha=0.06, color=TRAIL_COLOR, zorder=0)
ax1.axvspan( 0, 1, alpha=0.06, color=FWD_COLOR,   zorder=0)
ax1.set_xlabel('cos(θ$_{Pt}$ − θ$_{ld}$)', fontsize=10)
ax1.set_ylabel('PDF', fontsize=10)
ax1.tick_params(labelsize=8)
ax1.text( 0.7, ax1.get_ylim()[1] * 0.92, 'Pt forward',
          ha='center', color=FWD_COLOR,   fontsize=8, style='italic')
ax1.text(-0.7, ax1.get_ylim()[1] * 0.92, 'Pt trailing',
          ha='center', color=TRAIL_COLOR, fontsize=8, style='italic')
plt.tight_layout()
plt.show()


print(f"  mean cos = {cos_vals_pbs.mean():+.3f}")
print(f"mean resultant vector = {np.degrees(mu_rad_pbs):+.1f} deg")


In [ ]:
cos_all_pbs = np.asarray(pbs_cos_pooled, float)
n = cos_all_pbs.size
dth_rad = np.radians(np.concatenate(
    [np.asarray(r['dtheta'], float) for r in pbs_results if len(r['dtheta'])]))

mean_c = float(cos_all_pbs.mean())
sd_c   = float(cos_all_pbs.std(ddof=1))
sem_c  = sd_c / np.sqrt(n)
t_stat, p_t = stats.ttest_1samp(cos_all_pbs, 0.0, alternative='less')


tcrit = stats.t.ppf(0.975, n - 1)
ci_lo, ci_hi = mean_c - tcrit * sem_c, mean_c + tcrit * sem_c


V, u, p_v = vtest(dth_rad, np.pi)
Rbar   = float(np.abs(np.mean(np.exp(1j * dth_rad))))
mu_dir = float(np.degrees(np.angle(np.mean(np.exp(1j * dth_rad)))))

print(f"mean cos = {mean_c:+.3f}")
print(f"95% CI on mean = [{ci_lo:+.2f}, {ci_hi:+.2f}]")
print(f"one-sided t-test (H1: mean cos < 0): p = {p_t:.3f}")

print(f"mean direction = {mu_dir:+.1f}°")
print(f"V-test toward {np.degrees(np.pi):.0f}°: p = {p_v:.3f}")